In [1]:
import pdfplumber
import pandas as pd
import os

# Path to the directory containing the PDFs
pdf_directory = r"C:\1.WOW MOMO WORKING\8.2025-2026 Apps Reconciliation Report\1.All Apps Reconciliation\4.Swiggy\8.NOV-25\Ads_Inv"

# Columns you want to extract for the structured data
invoice_details_columns = ["Invoice Number :","Invoice Number ", "Invoice Date :", "Service Period :", "Invoice Type :", 
                           "Original Invoice No :", "Original Invoice", "Original Invoice Date", "Original Invoice Date :", "Legal Name :", "Address :", "State Code :"
                           "Restaurant / Store Name :", "Restaurant / Store ID :" ]

# Columns for the table under "Description"
table_columns = ["Sr. no.","Description", "HSN", "Unit of measure", "Quantity", "Unit Price", "Base Amount", "Discount", "Assessable value", "CGST Rate",
                 "CGST Amount", "SGST Rate", "SGST Amount", "IGST Rate", "IGST Amount", "Comp CESS Rate", "Comp CESS Amount", "State CESS Rate", 
                 "State CESS Amount", "Total Amount (Rs.)"]

# DataFrame to store all extracted data
extracted_data = []

# Function to extract text details from each PDF
def extract_details_and_table(pdf_path):
    with pdfplumber.open(pdf_path) as pdf:
        first_page = pdf.pages[0]
        text = first_page.extract_text()

        # Extracting the key invoice details
        invoice_details = {}
        for column in invoice_details_columns:
            if column in text:
                # Extract the value after the label
                start = text.find(column) + len(column)
                end = text.find('\n', start)
                value = text[start:end].strip()
                invoice_details[column] = value
            else:
                invoice_details[column] = None

        # Extracting table data (assuming it's on the first page)
        for page in pdf.pages:
            table = page.extract_table()
            if table:
                # Handle if the number of columns is not matching
                table_df = pd.DataFrame(table[1:], columns=table[0])  # Create DataFrame from table
                
                # Check if the extracted table has the expected number of columns
                if len(table_df.columns) >= len(table_columns):
                    # Trim or match the columns
                    table_df = table_df.iloc[:, :len(table_columns)]  # Keep only required columns
                    table_df.columns = table_columns  # Assign column names
                    
                    # **Filter the rows to keep only those where 'Unit of measure' is 'OTH'**
                    table_df = table_df[table_df['Unit of measure'] == 'OTH']
                else:
                    print(f"Warning: Table in {pdf_path} does not match expected column count.")
                    # Fill in missing columns with None if less columns are found
                    for i in range(len(table_columns) - len(table_df.columns)):
                        table_df[f'Missing Column {i}'] = None
                    table_df.columns = table_columns  # Assign the expected column names
                
                return invoice_details, table_df

        # Return the invoice details and empty DataFrame if no table is found
        return invoice_details, pd.DataFrame(columns=table_columns)

# Loop through all PDFs in the directory and extract data
for pdf_file in os.listdir(pdf_directory):
    if pdf_file.endswith('.pdf'):
        pdf_path = os.path.join(pdf_directory, pdf_file)
        details, table = extract_details_and_table(pdf_path)

        # Append the invoice details and the table data row-wise
        for _, row in table.iterrows():
            extracted_data.append({**details, **row.to_dict()})

# Convert the extracted data to a DataFrame
df = pd.DataFrame(extracted_data)

# Save the DataFrame to Excel
df.to_csv(r'C:\1.WOW MOMO WORKING\8.2025-2026 Apps Reconciliation Report\1.All Apps Reconciliation\4.Swiggy\8.NOV-25\Ads_Inv\1.Swiggy_Commission_extracted_invoice_data_.csv', index=False)
print("Data extraction complete and saved to 'extracted_invoice_data_filtered.CSV'")


Data extraction complete and saved to 'extracted_invoice_data_filtered.CSV'


In [2]:
pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install pdfplumber

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
